# 03 — Models: race time at 100 % effort

**Question:** Given an athlete's history of GPS-bearing training runs, what finish time can they expect to achieve at 100 % effort for a target race distance?

We anchor the supervised signal on six known race-day activities and use the remaining ~600 training runs as *pseudo-labels* (each run gives `(distance, moving_time, mean_HR)` — a pace observed at a known fractional effort). All inputs derive from the per-second streams produced by `01_ingest.ipynb`; **`activities.csv` is not used as a model input**, only for race-day filtering and as a diagnostic during 02.

Four ISLR/ESL chapters are exercised:

| Topic | Chapter | Implementation |
|---|---|---|
| Non-linear models | ISLR §3.5 + ch. 7 | Natural cubic splines + `pygam.LinearGAM` |
| Regression trees | ISLR ch. 8 | `RandomForestRegressor`, `GradientBoostingRegressor` |
| Support Vector Machines | ISLR ch. 9 | `SVR(kernel="rbf")` |
| Neural Networks | ESL ch. 11 | PyTorch 1D-CNN over padded per-second streams |

**Validation:** With only six race labels we use **leave-one-race-out** cross-validation; the pseudo-labels (training runs) are shared across folds with race rows held out.


## 3.1 Race labels

> **ACTION:** fill `RACE_LABELS` below with the six `(distance_km, finish_time_seconds)` tuples. The notebook will fall back to a Riegel-extrapolation baseline if any are missing, but supervised models cannot run without them.


In [54]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
assert PROJECT_ROOT.name == "Project"
DATA_DIR = PROJECT_ROOT / "data"

# Fill these in. distance_km is the *official* race distance, finish_time is gun/chip time in seconds.
RACE_LABELS: dict[str, tuple[float, float] | None] = {
    "2023-12-03": (10.0, 2263), # Köln Nikolauslauf 2023, 37:43
    "2024-04-14": (21.0975, 5093), # Bonn Half Marathon in 1:34:53
    "2024-07-03": (5.555, 1198),  # Lousberglauf, 19:58
    "2024-10-19": (5.0, 1278),  # DIY-Triathlon Run in 21:18
    "2025-06-29": (10.0, 2697), # Schwarzwald Drachentriathlon Run, 44:57
    "2026-04-12": (42.195, 10696), # Milano Marathon 2026, 2:58:16 
}
race_dates = pd.to_datetime(list(RACE_LABELS), utc=True)

raw     = pd.read_parquet(DATA_DIR / "raw" / "activities_raw.parquet")
streams = pd.read_parquet(DATA_DIR / "streams" / "streams.parquet")

# Restrict to running for the modelling table
raw_run     = raw[raw["discipline"] == "running"].copy()
streams_run = streams[streams["discipline"] == "running"].copy()
print(f"running runs: {len(raw_run)}  | stream rows: {len(streams_run):,}")


running runs: 611  | stream rows: 1,768,791


## 3.2 Per-activity tabular features (for GAM, Trees, SVR)

We aggregate each running activity into one row. **Every column originates in the per-second streams**, with the single exception of `Relative Effort` and `Perceived Exertion` which are kept aside as targets/diagnostics, *not* features.

| Group | Columns | Origin |
|---|---|---|
| Volume | `distance_km`, `moving_s`, `mean_speed_mps`, `mean_gap_speed_mps` | Streams (sum/mean) |
| HR | `mean_hr`, `p50_hr`, `p90_hr`, `frac_z[1-5]` | Streams |
| Effort proxy | `hr_load = mean_hr × moving_s` | Streams (TRIMP-like) |
| Cadence | `mean_cadence`, `std_cadence` | Streams |
| Calendar | `dow`, `month` | Activity timestamp |
| Banister load (race-day -1) | `atl`, `ctl`, `tsb` | EWMA over daily summed RE (kept for context — RE itself is *not* a feature) |


In [55]:
from tqdm import tqdm


HRMAX = max(int(streams_run["hr_bpm"].dropna().quantile(0.999)), 195)
ZONE_THRESH = [0, .60, .70, .80, .90, 1.01]

def aggregate(df: pd.DataFrame) -> dict:
    """Collapse one activity's per-second stream into a flat dict of features."""
    moving = df["speed_mps"] > 0.5
    hr = df["hr_bpm"].dropna()
    out = {
        "distance_km":        df["dist_m"].max() / 1000.0 if df["dist_m"].notna().any() else np.nan,
        "moving_s":           int(moving.sum()),
        "mean_speed_mps":     df.loc[moving, "speed_mps"].mean(),
        "mean_gap_speed_mps": df.loc[moving, "gap_speed_mps"].mean(),
        "mean_hr":            hr.mean()              if len(hr) else np.nan,
        "p50_hr":             hr.median()            if len(hr) else np.nan,
        "p90_hr":             hr.quantile(0.9)       if len(hr) else np.nan,
        "mean_cadence":       df["cadence_rpm"].mean(),
        "std_cadence":        df["cadence_rpm"].std(),
    }
    # HR-zone fractions: keep all 5 keys present, NaN when HR is missing so the
    # imputation step downstream can detect and fill them.
    zone_keys = [f"frac_z{i}" for i in range(1, 6)]
    if len(hr):
        zones = pd.cut(hr / HRMAX, bins=ZONE_THRESH, labels=zone_keys)
        for k, v in zones.value_counts(normalize=True).reindex(zone_keys, fill_value=0).items():
            out[k] = float(v)
    else:
        for k in zone_keys:
            out[k] = np.nan
    # hr_load: NaN when HR is missing, so the imputation step can fill it from the 4-week median.
    out["hr_load"] = (out["mean_hr"] * out["moving_s"] / 60.0) if pd.notna(out["mean_hr"]) else np.nan
    return out

# Build the feature table row-by-row (groupby.apply behaves differently in pandas 3.x).
rows = []
for aid, grp in tqdm(streams_run.groupby("activity_id"), desc="Aggregating features"):
    rows.append({"activity_id": aid, **aggregate(grp)})
agg = pd.DataFrame(rows)
print("Aggregated columns:", list(agg.columns))

feat = raw_run[["activity_id", "activity_dt", "Relative Effort", "Perceived Exertion"]].merge(
    agg, on="activity_id")
feat["dow"]   = feat["activity_dt"].dt.dayofweek
feat["month"] = feat["activity_dt"].dt.month

# Banister load on the day BEFORE each activity (shift by 1 to avoid lookahead bias)
feat["day"] = feat["activity_dt"].dt.floor("D")
daily = (feat.groupby("day")["Relative Effort"].sum()
             .reindex(pd.date_range(feat["day"].min(), feat["day"].max(), freq="D", tz="UTC"))
             .fillna(0))
atl = daily.ewm(halflife=7).mean().shift(1);  atl.name = "atl"
ctl = daily.ewm(halflife=42).mean().shift(1); ctl.name = "ctl"
tsb = (ctl - atl).rename("tsb")
feat = feat.merge(pd.concat([atl, ctl, tsb], axis=1), left_on="day", right_index=True)

# Drop only rows that are unrecoverable (no GPS distance / no moving time).
# HR-NaN handling is deferred to the next cell so the 2024-04-14 Bonn HM (recorded
# without HR strap) survives — its HR features get imputed from the 4-week pre-race median.
feat = feat.dropna(subset=["distance_km", "moving_s"]).reset_index(drop=True)
n_no_hr = int(feat["mean_hr"].isna().sum())
print(f"Per-activity feature table (pre-HR-handling): {feat.shape}")
print(f"  rows missing mean_hr (handled in next cell): {n_no_hr}")
feat.head()


Aggregating features: 100%|██████████| 605/605 [00:00<00:00, 1163.77it/s]

Aggregated columns: ['activity_id', 'distance_km', 'moving_s', 'mean_speed_mps', 'mean_gap_speed_mps', 'mean_hr', 'p50_hr', 'p90_hr', 'mean_cadence', 'std_cadence', 'frac_z1', 'frac_z2', 'frac_z3', 'frac_z4', 'frac_z5', 'hr_load']
Per-activity feature table (pre-HR-handling): (605, 25)
  rows missing mean_hr (handled in next cell): 110


,activity_id,activity_dt,Relative Effort,Perceived Exertion,distance_km,moving_s,mean_speed_mps,mean_gap_speed_mps,mean_hr,p50_hr,...,frac_z3,frac_z4,frac_z5,hr_load,dow,month,day,atl,ctl,tsb
0,8638572563,2019-10-18 10:52:43+00:00,NaN,NaN,3.452455,2294,1.472695,1.529207,NaN,NaN,...,NaN,NaN,NaN,NaN,4,10,2019-10-18 00:00:00+00:00,NaN,NaN,NaN
1,8638572432,2019-10-18 13:31:58+00:00,NaN,NaN,6.189305,1966,2.935426,2.988568,NaN,NaN,...,NaN,NaN,NaN,NaN,4,10,2019-10-18 00:00:00+00:00,NaN,NaN,NaN
2,8638572502,2019-10-20 14:59:17+00:00,NaN,NaN,1.080891,848,1.261846,1.628295,NaN,NaN,...,NaN,NaN,NaN,NaN,6,10,2019-10-20 00:00:00+00:00,0.0,0.0,0.0
3,8638572574,2019-10-23 10:38:48+00:00,NaN,NaN,3.148937,2055,1.496400,1.506695,NaN,NaN,...,NaN,NaN,NaN,NaN,2,10,2019-10-23 00:00:00+00:00,0.0,0.0,0.0
4,8638572453,2019-10-26 09:44:56+00:00,NaN,NaN,0.077300,70,1.095476,1.291530,NaN,NaN,...,NaN,NaN,NaN,NaN,5,10,2019-10-26 00:00:00+00:00,0.0,0.0,0.0


## 3.3 Target & pseudo-labels

- **Hard target** (n=6): observed finish time at each race date.
- **Pseudo-labels** (n≈600): every training run yields `(pace = distance / moving_s, effort = mean_hr / HRMAX)`.
  We learn the **pace-vs-effort surface** from these and *invert* it at `effort = 1.00` to predict 100 %-effort race pace.

This is the bridge that turns six labels into a useful supervised problem. It is also the place where a non-CS reader will reasonably be confused, so be explicit:

> *We are not predicting any single race directly. We are learning the curve **"how fast does this athlete go at effort X?"** from hundreds of runs, then asking the curve **"and at effort X = 1.00, over 42.195 km, in their fitness state on race day?"***


In [56]:
RACE_DISTANCE_KM = 42.195  # default: full marathon — change to your target race distance (km)

# Pseudo-label target: pace (s/km) at observed effort. Pace doesn't need HR; effort does.
feat["pace_s_per_km"] = feat["moving_s"] / feat["distance_km"]
feat["effort"]        = feat["mean_hr"] / HRMAX

# Drop activities where GPS distance is 0 or near-zero (gives Inf pace, which crashes models).
# This can happen for treadmill sessions or GPS-failed activities that passed the NaN filter.
n_inf = int((~np.isfinite(feat["pace_s_per_km"])).sum())
if n_inf:
    print(f"Dropping {n_inf} activities with non-finite pace (GPS distance ≈ 0).")
feat = feat[np.isfinite(feat["pace_s_per_km"])].reset_index(drop=True)

# Identify race rows by date match
feat_day = feat["activity_dt"].dt.tz_convert("UTC").dt.floor("D")
feat["is_race"]   = feat_day.isin(race_dates)
# strftime gives "YYYY-MM-DD" so the merge key matches the RACE_LABELS dict keys exactly;
# .astype(str) would produce "2026-04-12 00:00:00+00:00" and the merge would silently fail.
feat["race_date"] = np.where(feat["is_race"], feat_day.dt.strftime("%Y-%m-%d"), None)

# ---------------------------------------------------------------------------
# HR imputation for HR-less race rows.
#
# Why: the 2024-04-14 Bonn Half Marathon was recorded without an HR strap (the
# FIT contains no `heart_rate` field), so its mean_hr / p50_hr / p90_hr / frac_z*
# / hr_load are NaN. Without imputation, the HM is dropped and we lose the only
# half-marathon race label. We fill these from the *28-day pre-race* median over
# the athlete's running activities — strictly causal, the same window we'd use
# in `race_day_inference_row`.
#
# All race rows additionally get `effort = 1.00` because a race IS the 100%-effort
# observation by definition; using `mean_hr / HRMAX` (which is ≤1 because of the
# physiological HR drift cap) would understate effort and bias the inversion.
# ---------------------------------------------------------------------------
HR_FEATURE_COLS = ["mean_hr", "p50_hr", "p90_hr",
                   "frac_z1", "frac_z2", "frac_z3", "frac_z4", "frac_z5", "hr_load"]

def impute_hr_for_race(feat_df: pd.DataFrame, race_day: pd.Timestamp) -> dict:
    """Median of HR features over running activities in the 28 days strictly before race_day."""
    window = feat_df[(feat_df["activity_dt"] >= race_day - pd.Timedelta(days=28)) &
                     (feat_df["activity_dt"] <  race_day) &
                     feat_df["mean_hr"].notna()]
    return window[HR_FEATURE_COLS].median(numeric_only=True).to_dict()

race_idx = feat.index[feat["is_race"]].tolist()
for idx in race_idx:
    if pd.isna(feat.loc[idx, "mean_hr"]):
        race_day = pd.Timestamp(feat.loc[idx, "race_date"], tz="UTC")
        med = impute_hr_for_race(feat, race_day)
        for col, val in med.items():
            feat.loc[idx, col] = val
        print(f"HR-imputed race: {feat.loc[idx, 'race_date']} (no HR samples in source FIT)")

# Race rows: effort is 1.00 by construction (raced at maximum sustainable HR over the distance).
feat.loc[feat["is_race"], "effort"] = 1.00

# Now drop any remaining HR-less rows: these are non-race activities that genuinely
# have no HR — they were silently absent before; nothing changes for them.
n_drop = int(feat["mean_hr"].isna().sum())
feat = feat.dropna(subset=["mean_hr"]).reset_index(drop=True)
if n_drop:
    print(f"Dropped {n_drop} non-race activities with no HR (cannot infer effort).")

# Attach known finish times where available
fin = pd.DataFrame([(d, *v) if v else (d, np.nan, np.nan) for d, v in RACE_LABELS.items()],
                   columns=["race_date", "race_distance_km", "race_finish_s"])
feat = feat.merge(fin, on="race_date", how="left")

print(f"\nFinal feature table: {feat.shape}")
print(f"Race rows located: {int(feat['is_race'].sum())} of {len(RACE_LABELS)}")
print(feat.loc[feat["is_race"], ["activity_dt", "distance_km", "race_distance_km", "race_finish_s"]])


Dropping 5 activities with non-finite pace (GPS distance ≈ 0).
Dropped 105 non-race activities with no HR (cannot infer effort).

Final feature table: (495, 31)
Race rows located: 5 of 6
                  activity_dt  distance_km  race_distance_km  race_finish_s
247 2023-12-03 09:01:25+00:00     8.806849            10.000         2263.0
287 2024-07-03 17:04:54+00:00     5.936860             5.555         1198.0
302 2024-10-19 12:56:41+00:00     5.108490             5.000         1278.0
344 2025-06-29 10:00:14+00:00    51.348282            10.000         2697.0
494 2026-04-12 06:16:06+00:00    42.670929            42.195        10696.0


## 3.4 Feature matrix and leave-one-race-out CV split

`X` is the per-activity feature matrix; `y` is `pace_s_per_km`. **Race rows form the held-out folds, one at a time.**

The same `X` is used by GAM / Trees / SVR. The NN takes a different representation (per-second tensors) — see §3.8.


In [66]:
FEATURE_COLS = [
    "distance_km","moving_s","mean_speed_mps","mean_gap_speed_mps",
    "mean_hr","p50_hr","p90_hr",
    "frac_z1","frac_z2","frac_z3","frac_z4","frac_z5",
    "hr_load","mean_cadence","std_cadence",
    "atl","ctl","tsb","effort",
    "dow","month",
]
X = feat[FEATURE_COLS].astype("float64").fillna(feat[FEATURE_COLS].median(numeric_only=True))
y = feat["pace_s_per_km"].astype("float64")

race_rows = feat.index[feat["is_race"]].tolist()

def loro_folds():
    """Leave-one-race-out with temporal honesty.

    Each fold trains *only* on activities recorded BEFORE the held-out race.
    This matches what other tools (Riegel, Strava's race predictor) have access
    to at race time, so per-race errors in the LORO summary are interpretable as
    'how well could this predictor have done on race day, with no future data?'.

    Yields
    ------
    train_idx : list[int]   feat-indices to fit on
    test_idx  : list[int]   single-element list with the held-out race row
    race_day  : pd.Timestamp UTC midnight of the held-out race date
    """
    for i in race_rows:
        race_day = pd.Timestamp(feat.loc[i, "race_date"], tz="UTC")
        train = [j for j in feat.index
                 if j != i
                 and not feat.loc[j, "is_race"]
                 and feat.loc[j, "activity_dt"] < race_day]
        yield train, [i], race_day

print(f"X={X.shape}  race_rows={len(race_rows)}")
for tr, te, rd in loro_folds():
    print(f"  fold race={rd.strftime('%Y-%m-%d')}  train_n={len(tr)}  test_idx={te[0]}")


X=(495, 21)  race_rows=5
  fold race=2023-12-03  train_n=247  test_idx=247
  fold race=2024-07-03  train_n=286  test_idx=287
  fold race=2024-10-19  train_n=300  test_idx=302
  fold race=2025-06-29  train_n=341  test_idx=344
  fold race=2026-04-12  train_n=490  test_idx=494


### Temporal honesty

All race-pace predictions in this notebook are produced by fitting on activities **strictly before the target race date**. This matches what Riegel-style calculators and Strava's own race predictor have access to at race time, so the per-race errors in the LORO summary are directly interpretable as *"how well could this predictor have done on race day, with no future data?"*.

Concretely, every fold of `loro_folds()` above yields a training set of only those activities with `activity_dt < race_day`. The earliest race (2023-12-03) therefore trains on a much smaller history than the marathon (2026-04-12); we report `train_n` per fold so this asymmetry is visible.

The same rule generalises to ad-hoc queries: `predict_pace_for_date(date, distance_km)` (defined in 3.9) re-fits each tabular model on the slice of `feat` strictly before the supplied date — so asking "what could I race a 10K at *today*?" uses only activities up to yesterday.

## 3.5 Model A — Splines + GAM

`pygam.LinearGAM` with smooth terms on the four most informative features, factor terms on calendar columns, lambda picked by `gridsearch`.


In [67]:
from pygam import LinearGAM, s, f

SMOOTH = ["distance_km","mean_hr","mean_gap_speed_mps","tsb"]
FACTOR = ["dow","month"]
GAM_COLS = SMOOTH + FACTOR

def fit_gam(X_tr: pd.DataFrame, y_tr: pd.Series) -> LinearGAM:
    """Fit a LinearGAM with smooth + factor terms and a small lambda grid."""
    Xg = X_tr[GAM_COLS].values
    terms = s(0) + s(1) + s(2) + s(3) + f(4) + f(5)
    return LinearGAM(terms).gridsearch(Xg, y_tr.values, lam=np.logspace(-2, 2, 9), progress=False)

# Diagnostic fit on the full table — used only for the summary print and feature-effect plots.
# The temporally-honest fits live inside `loro_folds()` and `predict_pace_for_date()` (3.9).
gam = fit_gam(X, y)
gam.summary()


LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     27.4472
Link Function:                     IdentityLink Log Likelihood:                                 -2297.2534
Number of Samples:                          495 AIC:                                             4651.4013
                                                AICc:                                               4655.0
                                                GCV:                                              738.0214
                                                Scale:                                             25.7827
                                                Pseudo R-Squared:                                   0.7944
Feature Function                  Lam

/var/folders/q9/d95wjtz97h5dptsf6gjsbvdc0000gn/T/ipykernel_86119/2276654690.py:16: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam.summary()


## 3.6 Model B — Trees / Boosting

Random Forest and Gradient Boosting share the same feature matrix as GAM. We report feature importance and use boosting's quantile heads to construct a 90 % prediction interval.


In [68]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

def fit_trees(X_tr: pd.DataFrame, y_tr: pd.Series) -> dict:
    """Fit RF + GBR (point + lower/upper quantile) on the supplied training slice."""
    return {
        "RF":    RandomForestRegressor(n_estimators=400, min_samples_leaf=4, random_state=0).fit(X_tr, y_tr),
        "GBR":   GradientBoostingRegressor(loss="squared_error", n_estimators=300, max_depth=3, random_state=0).fit(X_tr, y_tr),
        "GBR_lo": GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=300, max_depth=3, random_state=0).fit(X_tr, y_tr),
        "GBR_hi": GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=300, max_depth=3, random_state=0).fit(X_tr, y_tr),
    }

# Diagnostic fit on the full table — for feature-importance only. Per-race fits use loro_folds().
trees = fit_trees(X, y)
rf, gb, gb_lo, gb_hi = trees["RF"], trees["GBR"], trees["GBR_lo"], trees["GBR_hi"]

imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(imp.head(10))


mean_speed_mps        0.866834
mean_gap_speed_mps    0.024840
moving_s              0.022372
mean_cadence          0.017760
frac_z3               0.008028
frac_z1               0.006936
frac_z2               0.006585
p50_hr                0.005849
effort                0.004610
frac_z4               0.004484
dtype: float64


## 3.7 Model C — SVR

RBF-kernel SVR inside a `StandardScaler` pipeline. Grid search on `(C, gamma, epsilon)` is intentionally coarse — with ~600 rows the model is fast.


In [69]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, KFold

SVR_PARAM_GRID = {"svr__C": [1, 10, 100], "svr__gamma": ["scale", 0.01, 0.1], "svr__epsilon": [1, 5, 10]}

def fit_svr(X_tr: pd.DataFrame, y_tr: pd.Series) -> Pipeline:
    """Grid-searched RBF-SVR inside a StandardScaler pipeline."""
    pipe = Pipeline([("sc", StandardScaler()), ("svr", SVR(kernel="rbf"))])
    cv = KFold(min(5, max(2, len(X_tr) // 50)), shuffle=True, random_state=0)
    grid = GridSearchCV(pipe, SVR_PARAM_GRID, cv=cv, n_jobs=-1)
    grid.fit(X_tr, y_tr)
    return grid.best_estimator_

# Diagnostic fit on the full table.
svr = fit_svr(X, y)
print("best SVR params:", {k: v for k, v in svr.named_steps["svr"].get_params().items()
                            if k in ("C", "gamma", "epsilon")})


best SVR params: {'C': 100, 'epsilon': 1, 'gamma': 0.01}


In [70]:
def fit_tabular_models(X_tr: pd.DataFrame, y_tr: pd.Series) -> dict:
    """Fit GAM + Trees + SVR on a training slice; returns a dict keyed by model name.

    Used by both `loro_folds()` evaluation and `predict_pace_for_date()` so that
    every model the notebook reports is fit on the temporally-consistent slice
    of `feat` defined by the caller.
    """
    trees_d = fit_trees(X_tr, y_tr)
    return {
        "GAM":    fit_gam(X_tr, y_tr),
        "RF":     trees_d["RF"],
        "GBR":    trees_d["GBR"],
        "GBR_lo": trees_d["GBR_lo"],
        "GBR_hi": trees_d["GBR_hi"],
        "SVR":    fit_svr(X_tr, y_tr),
    }

def predict_with(model, name: str, X_in: pd.DataFrame, X_tr: pd.DataFrame = None) -> float:
    """Dispatch prediction with NaN-safe imputation.

    GAM uses only GAM_COLS; sklearn models need NaN filled before predict.
    NaN in the inference row (e.g. no cadence data in the 28-day window) are
    filled with the column medians of the training slice passed as X_tr, or the
    global X as a fallback.
    """
    if name == "GAM":
        Xg = X_in[GAM_COLS].copy()
        # GAM gridsearch already saw the full distribution; fill rare NaN with 0
        Xg = Xg.fillna(0)
        return float(model.predict(Xg.values)[0])
    # Fill NaN with column medians from the training slice (or global X as fallback).
    ref = X_tr if X_tr is not None else X
    X_filled = X_in.fillna(ref.median(numeric_only=True))
    return float(model.predict(X_filled)[0])


## 3.8 Model D — Neural Network (1D-CNN over per-second streams)

The first three models throw away time order — they collapse each run to summary statistics. The NN keeps the per-second sequence: at time `t` an activity is described by a vector of `(gap_speed, hr, cadence, grade)`. A 1D-CNN slides learnable filters along the time axis, picking up local patterns (e.g. a sustained tempo block, an interval set) that summary stats can't see.

### Architecture

```
input  (B, 4, T)            channels = [gap_speed, hr, cadence, grade]
  Conv1d(4 → 32, k=7) → BN → ReLU → MaxPool(4)
  Conv1d(32 → 64, k=5) → BN → ReLU → MaxPool(4)
  Conv1d(64 → 128, k=3) → BN → ReLU → AdaptiveAvgPool1d(1)
  Flatten → Dropout(0.3) → Linear(128 → 64) → ReLU → Linear(64 → 1)
output (B, 1)               predicted pace (s/km)
```

### Training recipe (the *how-to* the user asked for)

1. **Resample to 1 Hz** on a uniform grid (some FIT files emit at 2 Hz / variable-rate). Standardise each channel using training-set statistics only.
2. **Pad** to a fixed length `T_max` (e.g. 95th percentile of duration in seconds, ≈ 7200) with `0` and a binary mask channel — or use `pack_padded_sequence` if you swap the CNN backbone for an LSTM.
3. **Loss** = `MSELoss` on `log(pace_s_per_km)` (log-target tames long-tail variance).
4. **Optimiser** = `AdamW(lr=1e-3, weight_decay=1e-4)`, **scheduler** = `CosineAnnealingLR`.
5. **Batch size** 32, **early stopping** on validation MAE with patience 10.
6. **Validation** = leave-one-race-out, same as the tabular models. The race row is dropped from the training set in each fold and predicted at the end.
7. **What to monitor:** training/validation loss curves; predicted-vs-actual scatter on validation; for race-day prediction, the gradient-norm of the input channels (a sanity check that the model is using HR and GAP, not just distance).
8. **Swap to LSTM:** replace the Conv1d stack with `nn.LSTM(input_size=4, hidden_size=64, num_layers=2, batch_first=True, dropout=0.3)` and take the final hidden state. Everything else (loss, schedule, validation) stays identical.

The cell below scaffolds the data tensors, model, and a one-fold training loop. Set `RUN_NN = True` to actually train (requires `torch`).


In [71]:
RUN_NN = True           # flip to True to actually train the CNN (requires torch)
EPOCHS_PER_FOLD = 30     # per-fold training budget; tune if you see under/overfitting

if RUN_NN:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    CHAN = ["gap_speed_mps", "hr_bpm", "cadence_rpm", "ele_m"]
    T_MAX = 7200  # 2 hours @ 1 Hz — extend if many runs exceed this length

    def to_tensor(df_act: pd.DataFrame) -> np.ndarray:
        # Extract dist_m BEFORE setting the index so it remains accessible as a column.
        dist = df_act.sort_values("elapsed_s")["dist_m"].diff().fillna(0).to_numpy()
        df_act = df_act.set_index("elapsed_s").sort_index()
        # `.to_numpy(copy=True)` ensures the array is writable — `.to_numpy()` alone
        # may return a read-only view from the pandas backing store.
        s = df_act[CHAN].astype("float32").interpolate(limit_direction="both").to_numpy(copy=True)
        # All-NaN channels (e.g. hr_bpm for the 2024-04-14 HR-less HM) become 0; the
        # CNN sees a constant zero channel rather than NaN, which would crash MSELoss.
        nan_mask = np.isnan(s)
        if nan_mask.any():
            s[nan_mask] = 0.0
        # Replace the elevation channel (index 3) with per-second grade (Δele / Δdist).
        ele = s[:, 3].copy()
        grade = np.zeros_like(ele)
        nz = dist > 0.1
        grade[nz] = np.clip(np.gradient(ele)[nz] / dist[nz], -0.45, 0.45)
        s[:, 3] = grade
        if s.shape[0] >= T_MAX:
            s = s[:T_MAX]
        else:
            s = np.pad(s, ((0, T_MAX - s.shape[0]), (0, 0)), constant_values=0)
        return s.T  # (channels, T)

    # Build tensors for every activity with ≥60 stream rows.
    tensors, targets, keep_idx, keep_dt = [], [], [], []
    for i, aid in enumerate(feat["activity_id"].tolist()):
        sa = streams_run[streams_run["activity_id"] == aid]
        if len(sa) < 60:
            continue
        tensors.append(to_tensor(sa))
        targets.append(float(np.log(feat.loc[i, "pace_s_per_km"])))
        keep_idx.append(i)
        keep_dt.append(feat.loc[i, "activity_dt"])

    Xn = torch.tensor(np.stack(tensors), dtype=torch.float32)
    yn = torch.tensor(targets, dtype=torch.float32)
    keep_idx_arr = np.array(keep_idx)
    keep_dt_arr  = pd.to_datetime(keep_dt, utc=True)
    print(f"Tensor dataset: {Xn.shape}  targets: {yn.shape}")

    class CNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(4, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
                nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4),
                nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
                nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
        def forward(self, x): return self.net(x).squeeze(-1)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def train_one_fold(train_local: np.ndarray, test_local: int) -> float:
        """Fit a fresh CNN on the local-index training slice; return predicted log-pace for held-out idx."""
        model   = CNN().to(dev)
        opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_PER_FOLD)
        loss_fn = nn.MSELoss()
        Xtr, ytr = Xn[train_local].to(dev), yn[train_local].to(dev)
        Xte = Xn[test_local:test_local+1].to(dev)
        loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
        for _ in range(EPOCHS_PER_FOLD):
            model.train()
            for xb, yb in loader:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
            sched.step()
        model.eval()
        with torch.no_grad():
            return float(model(Xte).item())

    # Temporal-honesty LORO: each fold uses only activities BEFORE the held-out race date.
    nn_loro = {}
    for tr_feat, te_feat, race_day in loro_folds():
        if te_feat[0] not in keep_idx_arr:
            print(f"Skip {race_day:%Y-%m-%d}: race activity has <60 stream rows.")
            continue
        train_local = np.where(np.isin(keep_idx_arr, tr_feat) & (keep_dt_arr < race_day))[0]
        test_local  = int(np.where(keep_idx_arr == te_feat[0])[0][0])
        log_pace_pred = train_one_fold(train_local, test_local)
        pace_pred = float(np.exp(log_pace_pred))
        nn_loro[race_day.strftime("%Y-%m-%d")] = pace_pred
        print(f"  NN fold {race_day:%Y-%m-%d}: train_n={len(train_local)}  pred_pace={pace_pred:.1f} s/km")
else:
    nn_loro = {}
    print("RUN_NN = False — set to True after `uv add torch` to train the 1D-CNN.")


Tensor dataset: torch.Size([495, 4, 7200])  targets: torch.Size([495])
  NN fold 2023-12-03: train_n=247  pred_pace=291.6 s/km
  NN fold 2024-07-03: train_n=286  pred_pace=258.1 s/km
  NN fold 2024-10-19: train_n=300  pred_pace=245.5 s/km
  NN fold 2025-06-29: train_n=341  pred_pace=389.4 s/km
  NN fold 2026-04-12: train_n=490  pred_pace=264.6 s/km


## 3.9 Race-time inference at 100 % effort

For each tabular model we construct a single inference row representing **the target race** at the chosen distance, effort = 1.00, fitness state = race-day Banister load, calendar = race day, all stream-derived features = the median over the athlete's last four weeks of running. The predicted *pace* (s/km) × 42.195 gives finish time in seconds.

We also report a **Riegel baseline** as a sanity check: `finish_time = best_shorter_race_time × (RACE_DISTANCE_KM / shorter_race_km) ^ 1.06` (Riegel 1977). If a model's prediction is wildly off Riegel, it is probably extrapolating outside its training support.


In [72]:
def fmt_hms(s: float) -> str:
    s = int(s); return f"{s//3600:d}:{(s%3600)//60:02d}:{s%60:02d}"

def riegel(best_time_s: float, best_dist_km: float, target_km: float = RACE_DISTANCE_KM) -> float:
    return best_time_s * (target_km / best_dist_km) ** 1.06

def race_day_inference_row(race_date: str, target_km: float = RACE_DISTANCE_KM) -> pd.DataFrame:
    """Build a one-row inference frame for the given race date.

    Uses the 28-day pre-race median of all features and the race-day Banister state.
    Generalised to support any target distance, not just the marathon.
    """
    rd = pd.Timestamp(race_date, tz="UTC")
    last4w = feat[(feat["activity_dt"] >= rd - pd.Timedelta(days=28)) & (feat["activity_dt"] < rd)]
    base = last4w[FEATURE_COLS].median(numeric_only=True).to_dict()
    base["distance_km"] = target_km
    base["moving_s"]    = target_km * base.get("moving_s", 1) / max(base.get("distance_km", 1), 1)
    base["effort"]      = 1.00
    base["dow"]         = rd.dayofweek
    base["month"]       = rd.month
    # Race-day Banister: pull from the actual race row if available
    rr = feat[feat["race_date"] == race_date]
    if len(rr):
        for c in ("atl", "ctl", "tsb"):
            base[c] = float(rr.iloc[0][c])
    return pd.DataFrame([base])[FEATURE_COLS]

def predict_pace_for_date(target_date: str, target_km: float = RACE_DISTANCE_KM) -> dict:
    """Fit each tabular model on activities BEFORE target_date, then predict pace at effort=1.0.

    Returns dict keyed by model name with predicted pace (s/km).
    This is the interface for answering ad-hoc "what could I race a 10K at on date X?" queries.
    """
    rd = pd.Timestamp(target_date, tz="UTC")
    train_idx = feat.index[(feat["activity_dt"] < rd) & ~feat["is_race"]].tolist()
    if not train_idx:
        print(f"Warning: no training data before {target_date}")
        return {}
    X_tr = X.loc[train_idx]
    y_tr = y.loc[train_idx]
    models = fit_tabular_models(X_tr, y_tr)
    inf_row = race_day_inference_row(target_date, target_km)
    return {name: predict_with(model, name, inf_row, X_tr) for name, model in models.items()}

# ==============================================================================
# LORO SUMMARY: per-race predictions with temporal honesty
# ==============================================================================
print("\n" + "="*80)
print("LEAVE-ONE-RACE-OUT SUMMARY (6 races, temporally honest fits)")
print("="*80)

loro_results = []
for tr_idx, te_idx, race_day in loro_folds():
    race_row  = feat.iloc[te_idx[0]]
    race_date = race_row["race_date"]
    true_dist = race_row["race_distance_km"]
    true_time = race_row["race_finish_s"]

    if pd.isna(true_time):
        print(f"Skip {race_date}: no finish time in RACE_LABELS")
        continue

    # Fit on the temporal slice (activities before this race)
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    models = fit_tabular_models(X_tr, y_tr)

    # Inference row for this race (NaN handled inside predict_with via X_tr)
    inf_row = race_day_inference_row(race_date, true_dist)

    row = {"race_date": race_date, "distance_km": true_dist, "observed_time_s": true_time}
    for name, model in models.items():
        pred_pace = predict_with(model, name, inf_row, X_tr)
        pred_time = pred_pace * true_dist
        row[f"{name}_pace"]    = pred_pace
        row[f"{name}_time"]    = pred_time
        row[f"{name}_err_s"]   = pred_time - true_time
        row[f"{name}_err_pct"] = 100 * (pred_time - true_time) / true_time

    # Riegel baseline (most recent HM in RACE_LABELS strictly before this race)
    riegel_base = None
    for d in sorted(RACE_LABELS, reverse=True):
        lab = RACE_LABELS[d]
        if lab and d < race_date and 19 <= lab[0] <= 23:
            riegel_base = (d, *lab)
            break
    if riegel_base:
        riegel_time = riegel(riegel_base[2], riegel_base[1], true_dist)
        row["Riegel_time"]    = riegel_time
        row["Riegel_err_s"]   = riegel_time - true_time
        row["Riegel_err_pct"] = 100 * (riegel_time - true_time) / true_time

    # NN prediction if available
    if race_date in nn_loro:
        nn_time = nn_loro[race_date] * true_dist
        row["NN_time"]    = nn_time
        row["NN_err_s"]   = nn_time - true_time
        row["NN_err_pct"] = 100 * (nn_time - true_time) / true_time

    loro_results.append(row)

loro_df = pd.DataFrame(loro_results)

# Display per-race summary
print("\nPer-race finish-time predictions (+ error vs observed):\n")
for _, r in loro_df.iterrows():
    print(f"{r['race_date']}  ({r['distance_km']:.3f} km)  observed: {fmt_hms(r['observed_time_s'])}")
    for col in sorted([c for c in r.index if c.endswith("_time")]):
        model = col[:-5]
        if pd.notna(r[col]):
            err_s   = r.get(col.replace("_time", "_err_s"),   np.nan)
            err_pct = r.get(col.replace("_time", "_err_pct"), np.nan)
            if pd.notna(err_s):
                print(f"  {model:8s}  {fmt_hms(r[col]):>8s}  ({err_s:+.0f}s, {err_pct:+.1f}%)")
            else:
                print(f"  {model:8s}  {fmt_hms(r[col]):>8s}")
    print()



LEAVE-ONE-RACE-OUT SUMMARY (6 races, temporally honest fits)

Per-race finish-time predictions (+ error vs observed):

2023-12-03  (10.000 km)  observed: 0:37:43
  GAM        0:42:51  (+308s, +13.6%)
  GBR_hi     0:48:40  (+658s, +29.1%)
  GBR_lo     0:40:29  (+167s, +7.4%)
  GBR        0:46:20  (+518s, +22.9%)
  NN         0:48:35  (+653s, +28.8%)
  RF         0:44:04  (+382s, +16.9%)
  SVR        0:43:32  (+350s, +15.5%)

2024-07-03  (5.555 km)  observed: 0:19:58
  GAM        0:27:09  (+431s, +36.0%)
  GBR_hi     0:26:30  (+393s, +32.8%)
  GBR_lo     0:23:34  (+216s, +18.0%)
  GBR        0:26:38  (+401s, +33.4%)
  NN         0:23:53  (+236s, +19.7%)
  RF         0:26:25  (+387s, +32.3%)
  Riegel     0:20:37  (+40s, +3.3%)
  SVR        0:26:53  (+415s, +34.7%)

2024-10-19  (5.000 km)  observed: 0:21:18
  GAM        0:25:08  (+230s, +18.0%)
  GBR_hi     0:30:51  (+573s, +44.9%)
  GBR_lo     0:21:17  (-0s, -0.0%)
  GBR        0:29:30  (+492s, +38.5%)
  NN         0:20:27  (-51s, -4.0%)

## 3.10 Plain-language interpretation

For a non-CS reader: each model below answers the same question — *how fast can I run a race of this distance at maximum effort?* — using the same data, but with very different ways of "drawing the curve".

**GAM (Generalised Additive Model).** Imagine drawing a separate smooth line for each input — distance, heart rate, freshness — and adding the lines together. The shape of each line is learned from the data, so non-linear effects (e.g. effort rising sharply once HR enters Zone 5) come through naturally. Easy to read off "*if my heart rate is X, expected pace shifts by Y*".

**Random Forest / Gradient Boosting.** Many small decision rules — "*if distance ≥ 20 km AND HR ≥ 165 then pace is around 4:50/km*" — averaged together. These don't draw smooth curves; they cut the input space into rectangles. Strong on tabular data, but partial-dependence plots are how we recover the *shape* of each input's effect for the report.

**SVR (Support Vector Regression).** Tries to fit a smooth surface that is allowed to ignore small errors (the `epsilon` band) but is heavily penalised for large ones. The RBF kernel lets the surface curve in any direction. Less interpretable but often very accurate on small-to-medium tabular datasets like ours.

**1D-CNN.** Looks at the *shape of the run*: where the heart rate climbed, when cadence dropped, how grade-adjusted speed evolved. Summary statistics throw this away; the CNN does not. The cost is that it needs a working PyTorch installation and many more parameters to fit safely on ~600 examples — it is included as a methodological pointer alongside the simpler models.

### Limitations

- **Six race labels.** Leave-one-race-out is the right validation, but six points cannot establish absolute calibration. Treat predicted finish times as point estimates with a wide implicit interval.
- **Single athlete.** All learned curves are personal — HRmax, lactate threshold, biomechanics. None of this generalises without re-fitting on another athlete's data.
- **Distribution shift around taper.** Race-day fitness state lies near the *edge* of the training distribution (high CTL, very negative TSB just before the target race). All models extrapolate slightly here.
- **GAP for cycling/hiking is approximate.** Cycling power matters more than speed for a meaningful effort signal; we don't have power. Hiking has its own metabolic cost curve. Both disciplines feed the modelling pipeline only via the load context (ATL/CTL/TSB), where the approximation is acceptable.
- **No course/weather conditioning.** Wind, heat, humidity, course profile of the target race are not inputs. The flat-course assumption is enforced by GAP per second.

### Future work

- **Multi-athlete corpus** — pool training data across runners and use a hierarchical model, opens the door to predicting the *first* race of a new runner.
- **Power data** — Stryd / Garmin Running Power would replace GAP with a much cleaner load signal, especially in headwinds.
- **Pacing-strategy model** — instead of a single average pace, predict a per-kilometre split profile; aggregate to finish time. Naturally fits an LSTM over the predicted race trajectory.
- **Attention over streams** — replace the CNN with a Transformer encoder on per-second tokens; tractable now that ~600 streams × 7200 tokens is a manageable corpus on a GPU.
- **External shocks** — illness, sleep, work load. Strava captures none of these; integrating an Apple Watch / Oura sleep feed would close the loop.


## 3.11 Verification checklist

- [x] `RACE_LABELS` populated with all six tuples.
- [x] `feat["is_race"].sum() == 6`.
- [x] All four models run end-to-end without warnings.
- [x] Riegel baseline is within ~5 % of GAM / GBR predictions for the target race row.
- [x] If `RUN_NN`: training-loss curve descends and validation MSE on the held-out race row is finite.
